In [21]:
import starfile
import dynamotable
import mrcfile
import pandas as pd
from pathlib import Path
from scipy.spatial.transform.rotation import Rotation as R

/tmp/ipykernel_44852/3596205805.py:6: DeprecationWarning: Please import `Rotation` from the `scipy.spatial.transform` namespace; the `scipy.spatial.transform.rotation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.spatial.transform.rotation import Rotation as R


In [22]:
tbl_file = "/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/dynamo_CJS_2025/alignment_projects/sirt_man550_best_workflow/top54_manual550_sirt_MaskOutAuNPsandMembrane_r6/results/ite_0001/averages/refined_table_ref_001_ite_0001.tbl"
#tbl_file = "/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/dynamo_CJS_2025/particle_tbls/bin1/bin1_all_sirt_extracted_crop_AngleRangesAdjusted.tbl"
tomo_doc = "/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/dynamo_CJS_2025/bin1_tomograms_sirt_top54.doc"

In [23]:
dynamo_data = dynamotable.read(tbl_file)

tomo_data = pd.read_csv(tomo_doc,delim_whitespace=True,header=None,names=['tomo','tomo_file'],dtype={'tomo':'int64'})
tomo_data['rlnTomoSizeX'] = 0
tomo_data['rlnTomoSizeY'] = 0
tomo_data['rlnTomoSizeZ'] = 0
for index, tomo_row in tomo_data.iterrows():
    tomo_path = Path("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS") / tomo_row['tomo_file']
    with mrcfile.open(tomo_path,mode='r',header_only=True) as mrc:
        tomo_data.at[index,'rlnTomoSizeX'] = mrc.header.nx
        tomo_data.at[index,'rlnTomoSizeY'] = mrc.header.ny
        tomo_data.at[index,'rlnTomoSizeZ'] = mrc.header.nz

print("Got tomo sizes")

combined = dynamo_data.merge(tomo_data,on='tomo',how='inner')
tomo_center_x = combined['rlnTomoSizeX'] / 2
tomo_center_y = combined['rlnTomoSizeY'] / 2
tomo_center_z = combined['rlnTomoSizeZ'] / 2
combined['rlnCoordinateX'] = ((combined['x'] + combined['dx'])) * 2.5
combined['rlnCoordinateY'] = ((combined['y'] + combined['dy'])) * 2.5
combined['rlnCoordinateZ'] = ((combined['z'] + combined['dz'])) * 2.5
combined['rlnAngleRot'] = combined['narot'] - 270
combined['rlnAngleTilt'] = - combined['tilt']
combined['rlnAnglePsi'] = combined['tdrot'] - 90
combined['CrossCorrelation'] = combined['cc']

# Fix the angular range
eulers = R.from_euler(seq="ZYZ", angles=combined[["rlnAngleRot","rlnAngleTilt","rlnAnglePsi"]].values, degrees=True)
combined["rlnAngleRot"] = eulers.as_euler("ZYZ", degrees=True)[:, 0]
combined["rlnAngleTilt"] = eulers.as_euler("ZYZ", degrees=True)[:, 1]
combined["rlnAnglePsi"] = eulers.as_euler("ZYZ", degrees=True)[:, 2]
combined['rlnTomoName'] = 'tomogram_' + combined['tomo'].astype(str)

relion_df = combined[['rlnCoordinateX','rlnCoordinateY','rlnCoordinateZ','rlnAngleRot','rlnAngleTilt','rlnAnglePsi','rlnTomoName', 'CrossCorrelation']]

print(relion_df)

#Relion location
#starfile.write({"particles":relion_df}, f"/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/relion5_2025/from_dynamo/{Path(tbl_file).stem}.star")

#ArtiaX location
starfile.write({"particles":relion_df}, f"/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/ArtiaX/tbls_from_dynamo/{Path(tbl_file).stem}.star")

/home/gouauxlab/andecath/env/lib/python3.11/site-packages/dynamotable/io.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(table_file, header=None, delim_whitespace=True)
/home/gouauxlab/andecath/env/lib/python3.11/site-packages/dynamotable/io.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors='ignore')
/tmp/ipykernel_44852/3409127669.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  tomo_data = pd.read_csv(tomo_doc,delim_whitespace=True,header=None,names=['tomo','tomo_file'],dtype={'tomo':'int64'})


Got tomo sizes
     rlnCoordinateX  rlnCoordinateY  rlnCoordinateZ  rlnAngleRot  \
0       5082.549275     5601.620000     1405.590000     -76.9700   
1       4067.732000     6656.600000     1959.467500    -115.4950   
2       4825.794000     5512.766500     1969.300750     -78.7480   
3       4775.747500     5435.719000     1822.187500     -25.1830   
4       5974.803500     3550.097500     1425.261550     164.6300   
..              ...             ...             ...          ...   
545     4867.989500     6219.963250     3083.296250     -63.3130   
546     4680.485000     5682.879375     2785.341425     -85.4145   
547     4710.901750     5524.308750     2624.122500    -107.2110   
548      354.065750     6350.481000     1559.016750      57.8900   
549     4020.660000     5947.483750     1100.666750     111.9600   

     rlnAngleTilt  rlnAnglePsi  rlnTomoName  CrossCorrelation  
0          94.637     135.6280   tomogram_1           0.39245  
1          79.863      97.5058   tomogra